In [ ]:
import pandas as pd
import missingno as msno
import geopandas as gpd
from shapely.geometry import Point
import rasterio
import matplotlib.pyplot as plt
import ezdxf
from rasterio.plot import show
import numpy as np
from rasterio.crs import CRS
import fiona

## 18 - Merge RegBL and Zones To Be Built

This notebooks' objective is to merge two datasets in their `processed` versions: 

- `17_ARE_zone_to_be_built`
- `03_OFS_regbl`

## Constants

In [ ]:
ROOT = '../../'

## Load the data

In [ ]:
df_regbl = pd.read_csv(f'{ROOT}data/processed/dataset-v0.14.csv', index_col=0)

df_regbl.head()

In [ ]:
gdf_tobebuilt = gpd.read_file(f"{ROOT}data/raw/17_ARE_zone_to_be_built/ch.are.bauzonen.gpkg")

gdf_tobebuilt.head()

In [ ]:
gdf_tobebuilt.crs

## Convert main DataFrame to GeoDataFrame

In [ ]:
df_regbl = gpd.GeoDataFrame(df_regbl, geometry=gpd.points_from_xy(df_regbl["Coordonnée E"], df_regbl["Coordonnée N"]), crs='EPSG:2056')

In [ ]:
columns = list(gdf_tobebuilt.columns)

# Add 'TBB - ' to the beginning of each column name except for 'geometry'
for column in columns:
    if column != 'geometry':
        gdf_tobebuilt = gdf_tobebuilt.rename(columns={column: f'TBB - {column}'})

## Perform spatial joint

In [ ]:
if "index_right" in df_regbl.columns:
    df_regbl = df_regbl.drop(columns="index_right")

In [ ]:
df_regbl = gpd.sjoin(df_regbl, gdf_tobebuilt, how="left", predicate="within")

In [ ]:
df_regbl.head()

In [ ]:
df_regbl.shape

In [ ]:
msno.matrix(df_regbl)

## Save the data

In [ ]:
df_regbl.to_csv(f"{ROOT}data/processed/dataset-v0.15.csv")